**Requirements (install locally):**
```bash
pip install psycopg2-binary SQLAlchemy geoalchemy2 geopandas shapely pyproj fiona rasterio matplotlib flask
```

**Conda environment (optional):**
```bash
conda create -n postgis python=3.10
conda activate postgis
conda install -c conda-forge psycopg2 sqlalchemy geoalchemy2 geopandas shapely pyproj fiona rasterio matplotlib flask
```


In [28]:
# 1) Connect to PostGIS
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

PG_USER = "sakdahomhuan"
PG_PASS = "1234"
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB   = "geo377"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    poolclass=NullPool, future=True
)

with engine.begin() as conn:
    version = conn.execute(text("SELECT postgis_full_version();")).scalar_one()
version


'POSTGIS="3.6.0 0" [EXTENSION] PGSQL="140" GEOS="3.14.0-CAPI-1.20.4" PROJ="9.6.2 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/Users/sakdahomhuan/Library/Application Support/proj DATABASE_PATH=/opt/homebrew/Cellar/proj/9.6.2/share/proj/proj.db" (compiled against PROJ 9.6.2) GDAL="GDAL 3.11.3 "Eganville", released 2025/07/12" LIBXML="2.13.8" LIBJSON="0.18" LIBPROTOBUF="1.5.2" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER'

In [29]:
# 2) Initialize schema
sql = open("python_17_postgis_setup.sql", "r", encoding="utf-8").read()
with engine.begin() as conn:
    conn.exec_driver_sql(sql)
print("PostGIS workshop schema initialized.")


PostGIS workshop schema initialized.


In [30]:
# 3) Checks counts and SRID
from pandas import DataFrame
from sqlalchemy import text
with engine.begin() as conn:
    rs = conn.execute(text("""

        SELECT 'hospital' AS tbl, COUNT(*) AS n, MIN(ST_SRID(geom)) AS srid FROM public.cm_hospital_4326

        UNION ALL

        SELECT 'districts' AS tbl, COUNT(*) AS n, MIN(ST_SRID(geom)) AS srid FROM workshop.districts;

    """))
    df = DataFrame(rs.fetchall(), columns=rs.keys())
df


,tbl,n,srid
0,hospital,59,4326
1,districts,1,4326


In [31]:
# 4) Loads data from PostGIS to GeoDataFrame
import geopandas as gpd
from shapely.geometry import Point

gpoi = gpd.read_postgis(
    "SELECT id, name, geom FROM public.cm_hospital_4326",
    engine, geom_col="geom"
)
display(gpoi.head())


,id,name,geom
0,1,โรงพยาบาลกรุงเทพเชียงใหม่,POINT (99.02642 18.78853)
1,2,โรงพยาบาลกองบิน 41,POINT (98.97138 18.77809)
2,3,โรงพยาบาลกาวิละ,POINT (99.01483 18.77794)
3,4,โรงพยาบาลค่ายกาวิละ,POINT (99.01488 18.77793)
4,5,โรงพยาบาลจอมทอง,POINT (98.67392 18.40632)


In [32]:
# 5) Spatial query: find POIs within districts
import pandas as pd
from sqlalchemy import text

q1 = text("""

SELECT p.id, p.name,

       ST_AsText(p.geom) AS wkt,

       ST_Within(p.geom, d.geom) AS inside_district

FROM public.cm_hospital_4326 p

JOIN workshop.districts d

  ON p.geom && d.geom AND ST_Intersects(p.geom, d.geom);

""")
with engine.begin() as conn:
    df = pd.DataFrame(conn.execute(q1).fetchall(), columns=["id","name","wkt","inside_district"])
df


,id,name,wkt,inside_district
0,6,โรงพยาบาลช้างเผือก,POINT(98.98704276000007 18.798882440000057),True
1,10,โรงพยาบาลเชียงใหม่ราม,POINT(98.97775677000004 18.795082310000055),True
2,18,โรงพยาบาลเด็กเชียงใหม่ราม,POINT(98.97761676000005 18.794845690000045),True
3,23,โรงพยาบาลเทพปัญญา2,POINT(98.98704010000006 18.79892756000004),True


In [33]:
# 6) Spatial query: KNN (K-Nearest Neighbor) search จาก hospital แต่ละแห่ง หา hospital ที่ใกล้ที่สุด 1 แห่ง
q2 = text("""

SELECT p.id, p.name,

       f.id  AS nearest_id,

       ST_Distance(p.geom::geography, f.geom::geography) AS meters

FROM public.cm_hospital_4326 p

CROSS JOIN LATERAL (

  SELECT id, geom FROM public.cm_hospital_4326 f

  WHERE f.id <> p.id

  ORDER BY f.geom <-> p.geom

  LIMIT 1

) f;

""")

with engine.begin() as conn:

    knn = conn.execute(q2).fetchall()

knn[:5]


[(1, 'โรงพยาบาลกรุงเทพเชียงใหม่', 20, 819.33034467),
 (2, 'โรงพยาบาลกองบิน 41', 49, 841.44439205),
 (3, 'โรงพยาบาลกาวิละ', 4, 5.00198492),
 (4, 'โรงพยาบาลค่ายกาวิละ', 3, 5.00198492),
 (5, 'โรงพยาบาลจอมทอง', 16, 13768.62303711)]

In [34]:
# 7) Spatial query: aggregate hospital counts by district
q3 = text("""

SELECT d.name, COUNT(p.*) AS cnt

FROM workshop.districts d

LEFT JOIN public.cm_hospital_4326 p

  ON d.geom && p.geom AND ST_Intersects(d.geom, p.geom)

GROUP BY d.name ORDER BY cnt DESC;

""")

import pandas as pd

with engine.begin() as conn:

    agg = pd.DataFrame(conn.execute(q3).fetchall(), columns=["district","count"])

agg


,district,count
0,Demo District,4


In [39]:
# 8) Export query result as GeoJSON
from sqlalchemy import text
q_geojson = text("""

SELECT jsonb_build_object(

  'type','FeatureCollection',

  'features', jsonb_agg(jsonb_build_object(

     'type','Feature',

     'properties', jsonb_build_object('id', id, 'name', name),

     'geometry', ST_AsGeoJSON(geom)::jsonb

  ))

) AS fc

FROM public.cm_hospital_4326;

""")

with engine.begin() as conn:

    fc = conn.execute(q_geojson).scalar_one()

fc


{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [99.02642032, 18.78853112]},
   'properties': {'id': 1, 'name': 'โรงพยาบาลกรุงเทพเชียงใหม่'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [98.97138436, 18.77808769]},
   'properties': {'id': 2, 'name': 'โรงพยาบาลกองบิน 41'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [99.01483439, 18.77794254]},
   'properties': {'id': 3, 'name': 'โรงพยาบาลกาวิละ'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [99.01487874, 18.77792649]},
   'properties': {'id': 4, 'name': 'โรงพยาบาลค่ายกาวิละ'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [98.67391898, 18.40632131]},
   'properties': {'id': 5, 'name': 'โรงพยาบาลจอมทอง'}},
  {'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [98.98704276, 18.79888244]},
   'properties': {'id': 6, 'name': 'โรงพยาบาลช้างเผือก'}},
  {'type

In [41]:
# show with folium
import folium
m = folium.Map(location=[18.79, 98.98], zoom_start=12)
# folium.GeoJson(fc, name="geojson").add_to(m)
# add marker popup 
for feature in fc['features']:
    coords = feature['geometry']['coordinates'][::-1]  # reverse to (lat, lon)
    name = feature['properties']['name']
    folium.Marker(location=coords, popup=name).add_to(m)

m
#  --- IGNORE ---   